# Week 10: Baseline Recommendation Systems

This notebook builds two baseline recommenders for the MovieLens 25M catalog.

Goal for this step:
- build a popularity-based baseline (global and cluster-aware)
- build a content-based baseline using cosine similarity on autoencoder embeddings
- define the candidate pool and evaluation protocol
- save all outputs for comparison in the evaluation notebook

Both baselines are required benchmarks. The advanced model must beat them on the same evaluation setup.

## Scope

This notebook covers the baseline layer of the Week 10 recommendation milestone:
- `popularity_global`: recommend the top-N most-rated movies from the full catalog
- `popularity_cluster`: recommend the top-N most-rated movies within the same cluster as the query movie
- `content_cosine`: for a query movie, retrieve the top-N nearest neighbors in the 13-dimensional autoencoder embedding space

Inputs:
- `artifacts/week07/week07_autoencoder_embeddings_latent_13.parquet` — embedding matrix
- `artifacts/week07/week07_kmeans_assignments.csv` — cluster labels
- `data/processed/week03_v1/movies_catalog.parquet` — title and metadata
- `data/processed/week03_v1/ratings_clean.parquet` — interaction data for popularity

In [14]:
from pathlib import Path

import json

import numpy as np
import pandas as pd
import polars as pl
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

project_root = Path.cwd()
if not (project_root / 'data').exists():
    project_root = project_root.parent
if not (project_root / 'data').exists():
    project_root = project_root.parent

ARTIFACTS_DIR = project_root / 'artifacts' / 'week10'
WEEK07_DIR = project_root / 'artifacts' / 'week07'
DATA_DIR = project_root / 'data' / 'processed' / 'week03_v1'
RANDOM_STATE = 42

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

required = [
    WEEK07_DIR / 'week07_autoencoder_embeddings_latent_13.parquet',
    WEEK07_DIR / 'week07_kmeans_assignments.csv',
    DATA_DIR / 'movies_catalog.parquet',
    DATA_DIR / 'ratings_clean.parquet',
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(f'Missing required inputs: {missing}')

ARTIFACTS_DIR

PosixPath('/Users/jay17/Documents/Proyects/big-data-tf/artifacts/week10')

## 1) Load inputs

We load the autoencoder embeddings, cluster assignments, movie catalog, and rating aggregates.
The rating aggregates are used both for popularity ranking and as context for evaluation.

In [15]:
# Load autoencoder embeddings (13-dimensional)
embeddings_pl = pl.read_parquet(WEEK07_DIR / 'week07_autoencoder_embeddings_latent_13.parquet')

# Load cluster assignments from Week 7 K-means (k=7)
clusters_pl = pl.read_csv(WEEK07_DIR / 'week07_kmeans_assignments.csv')

# Load catalog metadata
catalog_pl = pl.read_parquet(DATA_DIR / 'movies_catalog.parquet')

# Load ratings for popularity signal
ratings_pl = pl.read_parquet(DATA_DIR / 'ratings_clean.parquet')

summary = pl.DataFrame([
    {'table': 'embeddings', 'rows': embeddings_pl.height, 'cols': embeddings_pl.width},
    {'table': 'clusters',   'rows': clusters_pl.height,   'cols': clusters_pl.width},
    {'table': 'catalog',    'rows': catalog_pl.height,    'cols': catalog_pl.width},
    {'table': 'ratings',    'rows': ratings_pl.height,    'cols': ratings_pl.width},
])

summary

table,rows,cols
str,i64,i64
"""embeddings""",62423,14
"""clusters""",62423,2
"""catalog""",62423,9
"""ratings""",25000095,5


## 2) Build the master movie frame

Join catalog, cluster, embedding, and rating aggregates into a single working frame.
Only movies with at least one rating enter the evaluation pool.

In [16]:
# Compute rating aggregates at movie level
rating_agg = (
    ratings_pl
    .group_by('movieId')
    .agg([
        pl.len().alias('rating_count'),
        pl.col('rating').mean().alias('avg_rating'),
        pl.col('rating').std().alias('rating_std'),
        pl.col('userId').n_unique().alias('unique_raters'),
    ])
)

# Normalize clusters column name if needed
cluster_col = 'cluster' if 'cluster' in clusters_pl.columns else clusters_pl.columns[1]
clusters_renamed = clusters_pl.rename({cluster_col: 'cluster'}) if cluster_col != 'cluster' else clusters_pl

# Build master frame
movie_frame = (
    catalog_pl
    .select(['movieId', 'title', 'genres_list', 'release_year'])
    .join(rating_agg, on='movieId', how='left')
    .join(clusters_renamed.select(['movieId', 'cluster']), on='movieId', how='left')
    .filter(pl.col('rating_count').is_not_null() & (pl.col('rating_count') > 0))
    .with_columns([
        pl.col('cluster').fill_null(-1).cast(pl.Int32),
        pl.col('rating_count').fill_null(0).cast(pl.Int64),
        pl.col('avg_rating').fill_null(0.0),
    ])
    .sort('movieId')
)

print(f'Master frame: {movie_frame.height:,} movies with ratings')
print(f'Cluster distribution:')
display(movie_frame.group_by('cluster').agg(pl.len().alias('count')).sort('cluster').to_pandas())
movie_frame.head(5)

Master frame: 59,047 movies with ratings
Cluster distribution:


,cluster,count
0,0,9363
1,1,2650
2,2,5013
3,3,10036
4,4,23093
5,5,5346
6,6,3546


movieId,title,genres_list,release_year,rating_count,avg_rating,rating_std,unique_raters,cluster
i64,str,list[str],i64,i64,f64,f64,u32,i32
1,"""Toy Story (1995)""","[""Adventure"", ""Animation"", … ""Fantasy""]",1995,57309,3.893708,0.921552,57309,0
2,"""Jumanji (1995)""","[""Adventure"", ""Children"", ""Fantasy""]",1995,24228,3.251527,0.959851,24228,0
3,"""Grumpier Old Men (1995)""","[""Comedy"", ""Romance""]",1995,11804,3.142028,1.008443,11804,0
4,"""Waiting to Exhale (1995)""","[""Comedy"", ""Drama"", ""Romance""]",1995,2523,2.853547,1.108531,2523,0
5,"""Father of the Bride Part II (1…","[""Comedy""]",1995,11714,3.058434,0.996611,11714,0


## 3) Popularity baseline — global

The global popularity baseline ranks every movie by rating count (the simplest possible signal).
We use rating count rather than average rating because count captures genuine engagement:
a movie with 50,000 ratings at 3.8 is more meaningful as a recommendation than one with 5 ratings at 5.0.

We also compute a Bayesian-adjusted score (Wilson-like) that penalizes movies with very few ratings.

In [17]:
# Global mean rating for Bayesian adjustment
global_mean = float(movie_frame.select(pl.col('avg_rating').mean()).item())
# Prior count: use the 10th percentile of rating counts to avoid inflating niche films
prior_count = float(movie_frame.select(pl.col('rating_count').cast(pl.Float64).quantile(0.10)).item())

popularity_global = (
    movie_frame
    .with_columns([
        # Bayesian average: (C * m + n * r) / (C + n)
        (
            (prior_count * global_mean + pl.col('rating_count').cast(pl.Float64) * pl.col('avg_rating'))
            / (prior_count + pl.col('rating_count').cast(pl.Float64))
        ).alias('bayesian_score'),
    ])
    .select(['movieId', 'title', 'genres_list', 'release_year', 'rating_count', 'avg_rating', 'bayesian_score', 'cluster'])
    .sort('rating_count', descending=True)
)

# CSV does not support nested list columns — serialize genres_list to a pipe-joined string.
# The list column is preserved in-memory (popularity_global) for downstream operations.
popularity_global_csv = popularity_global.with_columns([
    pl.col('genres_list').list.join('|').alias('genres'),
]).drop('genres_list')
popularity_global_csv.write_csv(ARTIFACTS_DIR / 'week10_popularity_global.csv')

print(f'Global popularity: {popularity_global.height:,} ranked movies')
print(f'Global mean rating (μ): {global_mean:.4f}')
print(f'Prior count (C): {prior_count:.1f}')
display(popularity_global.head(10).to_pandas())

Global popularity: 59,047 ranked movies
Global mean rating (μ): 3.0714
Prior count (C): 1.0


,movieId,title,genres_list,release_year,rating_count,avg_rating,bayesian_score,cluster
0,356,Forrest Gump (1994),"[Comedy, Drama, Romance, War]",1994,81491,4.048011,4.047999,0
1,318,"Shawshank Redemption, The (1994)","[Crime, Drama]",1994,81482,4.413576,4.413560,0
2,296,Pulp Fiction (1994),"[Comedy, Crime, Drama, Thriller]",1994,79672,4.188912,4.188898,0
3,593,"Silence of the Lambs, The (1991)","[Crime, Horror, Thriller]",1991,74127,4.151342,4.151327,3
4,2571,"Matrix, The (1999)","[Action, Sci-Fi, Thriller]",1999,72674,4.154099,4.154084,0
5,260,Star Wars: Episode IV - A New Hope (1977),"[Action, Adventure, Sci-Fi]",1977,68717,4.120189,4.120173,0
6,480,Jurassic Park (1993),"[Action, Adventure, Sci-Fi, Thriller]",1993,64144,3.679175,3.679166,0
7,527,Schindler's List (1993),"[Drama, War]",1993,60411,4.247579,4.247560,1
8,110,Braveheart (1995),"[Action, Drama, War]",1995,59184,4.002273,4.002257,1
9,2959,Fight Club (1999),"[Action, Crime, Drama, Thriller]",1999,58773,4.228311,4.228291,0


## 4) Popularity baseline — cluster-aware

The cluster-aware popularity baseline ranks movies within the same Week 7 cluster as a query movie.
This is a stronger baseline than global popularity because it uses genre/style context from the clustering layer.

In [18]:
# Cluster-level popularity: rank within each cluster by bayesian_score
popularity_cluster = (
    popularity_global
    .with_columns([
        pl.col('bayesian_score')
        .rank(method='dense', descending=True)
        .over('cluster')
        .alias('cluster_rank'),
    ])
    .sort(['cluster', 'cluster_rank'])
)

# Serialize genres_list to pipe-joined string before writing CSV
popularity_cluster_csv = popularity_cluster.with_columns([
    pl.col('genres_list').list.join('|').alias('genres'),
]).drop('genres_list')
popularity_cluster_csv.write_csv(ARTIFACTS_DIR / 'week10_popularity_cluster.csv')

# Show top-5 per cluster
top5_per_cluster = (
    popularity_cluster
    .filter(pl.col('cluster_rank') <= 5)
    .select(['cluster', 'cluster_rank', 'movieId', 'title', 'rating_count', 'bayesian_score'])
)

print('Top-5 movies per cluster (cluster-aware popularity):')
display(top5_per_cluster.to_pandas())

Top-5 movies per cluster (cluster-aware popularity):


,cluster,cluster_rank,movieId,title,rating_count,bayesian_score
0,0,1,318,"Shawshank Redemption, The (1994)",81482,4.413560
1,0,2,858,"Godfather, The (1972)",52498,4.324312
2,0,3,171495,Cosmos,277,4.322199
3,0,4,50,"Usual Suspects, The (1995)",55366,4.284331
4,0,5,2019,Seven Samurai (Shichinin no samurai) (1954),13367,4.254681
...,...,...,...,...,...,...
62,6,1,179589,Windstorm 2 (2015),2,4.357125
63,6,2,196965,Fate/Stay Night: Unlimited Blade Works (2010),3,4.267843
64,6,3,204366,One Piece: Baron Omatsuri and the Secret Islan...,5,4.261896
65,6,4,163809,Over the Garden Wall (2013),546,4.256072


## 5) Content-based baseline — cosine similarity on embeddings

For each query movie, the content-based baseline retrieves the top-N most similar movies using
cosine similarity over the 13-dimensional autoencoder embedding space.

The embedding matrix is L2-normalized before computing similarities so that all dimensions
contribute equally regardless of scale.

In [19]:
# Align embeddings with the master frame (only movies with ratings)
embedding_cols = [c for c in embeddings_pl.columns if c != 'movieId']
rated_movie_ids = set(movie_frame['movieId'].to_list())

embeddings_rated = (
    embeddings_pl
    .filter(pl.col('movieId').is_in(list(rated_movie_ids)))
    .sort('movieId')
)

embedding_matrix = embeddings_rated.select(embedding_cols).to_numpy().astype(np.float32)
embedding_movie_ids = embeddings_rated['movieId'].to_list()
movie_id_to_idx = {mid: idx for idx, mid in enumerate(embedding_movie_ids)}

# L2-normalize rows for cosine similarity
embedding_normed = normalize(embedding_matrix, norm='l2')

print(f'Embedding matrix shape: {embedding_matrix.shape}')
print(f'Movies with embeddings AND ratings: {len(embedding_movie_ids):,}')

Embedding matrix shape: (59047, 13)
Movies with embeddings AND ratings: 59,047


In [20]:
def content_recommend(query_movie_id: int, top_n: int = 20, exclude_self: bool = True) -> list[dict]:
    """Return top-N content-based recommendations for a query movie.

    Uses precomputed L2-normalized embeddings and dot-product (= cosine similarity).
    The query movie itself is excluded from the result by default.
    """
    if query_movie_id not in movie_id_to_idx:
        return []

    query_idx = movie_id_to_idx[query_movie_id]
    query_vec = embedding_normed[query_idx].reshape(1, -1)  # shape (1, 13)

    # Dot product with all movies = cosine similarity (normalized)
    scores = (embedding_normed @ query_vec.T).flatten()

    # Sort descending, skip self
    sorted_idxs = np.argsort(-scores)
    results = []
    for idx in sorted_idxs:
        mid = embedding_movie_ids[idx]
        if exclude_self and mid == query_movie_id:
            continue
        results.append({'movieId': mid, 'cosine_similarity': float(scores[idx])})
        if len(results) >= top_n:
            break
    return results


# Demonstrate with well-known movies
demo_ids = [
    1,      # Toy Story (1995)
    296,    # Pulp Fiction (1994)
    318,    # Shawshank Redemption (1994)
    2571,   # Matrix (1999)
]

demo_rows = []
title_map = dict(zip(catalog_pl['movieId'].to_list(), catalog_pl['title'].to_list()))

for qid in demo_ids:
    recs = content_recommend(qid, top_n=5)
    for rank, rec in enumerate(recs, 1):
        demo_rows.append({
            'query_id': qid,
            'query_title': title_map.get(qid, '?'),
            'rank': rank,
            'rec_movieId': rec['movieId'],
            'rec_title': title_map.get(rec['movieId'], '?'),
            'cosine_similarity': round(rec['cosine_similarity'], 4),
        })

demo_df = pd.DataFrame(demo_rows)
print('Content-based recommendations (demo):')
display(demo_df)

Content-based recommendations (demo):


,query_id,query_title,rank,rec_movieId,rec_title,cosine_similarity
0,1,Toy Story (1995),1,4886,"Monsters, Inc. (2001)",0.9132
1,1,Toy Story (1995),2,8961,"Incredibles, The (2004)",0.8970
2,1,Toy Story (1995),3,4232,Spy Kids (2001),0.8887
3,1,Toy Story (1995),4,3751,Chicken Run (2000),0.8800
4,1,Toy Story (1995),5,410,Addams Family Values (1993),0.8749
5,296,Pulp Fiction (1994),1,50,"Usual Suspects, The (1995)",0.9218
6,296,Pulp Fiction (1994),2,57669,In Bruges (2008),0.9062
7,296,Pulp Fiction (1994),3,420,Beverly Hills Cop III (1994),0.8991
8,296,Pulp Fiction (1994),4,6709,Once Upon a Time in Mexico (2003),0.8883
9,296,Pulp Fiction (1994),5,786,Eraser (1996),0.8881


## 6) Precompute content-based recommendations for the evaluation set

For offline evaluation we precompute top-20 content-based recommendations for every movie
in the evaluation pool. Storing these avoids recomputing them in the evaluation notebook.

We use a sample of query movies for efficiency: the 5,000 most-rated movies.
These represent the core of the catalog and are most likely to appear in user histories.

In [21]:
TOP_N = 20
N_QUERY_MOVIES = 5000  # limit to top-rated movies for evaluation

# Select query movies: top-rated by count
query_movie_ids = (
    movie_frame
    .sort('rating_count', descending=True)
    .head(N_QUERY_MOVIES)
    ['movieId']
    .to_list()
)

print(f'Precomputing content-based recommendations for {len(query_movie_ids):,} query movies (top-N={TOP_N})...')

content_recs_rows = []
for qid in query_movie_ids:
    recs = content_recommend(qid, top_n=TOP_N)
    for rank, rec in enumerate(recs, 1):
        content_recs_rows.append({
            'query_movieId': qid,
            'rec_movieId': rec['movieId'],
            'rank': rank,
            'cosine_similarity': rec['cosine_similarity'],
        })

content_recs_pl = pl.DataFrame(content_recs_rows)
content_recs_pl.write_parquet(ARTIFACTS_DIR / 'week10_content_recs_top20.parquet')

print(f'Saved {content_recs_pl.height:,} recommendation rows.')
content_recs_pl.head(10)

Precomputing content-based recommendations for 5,000 query movies (top-N=20)...
Saved 100,000 recommendation rows.


query_movieId,rec_movieId,rank,cosine_similarity
i64,i64,i64,f64
356,260,1,0.909621
356,953,2,0.845615
356,4246,3,0.831167
356,111921,4,0.8122
356,7147,5,0.80449
356,192283,6,0.790146
356,1197,7,0.783587
356,77414,8,0.777833
356,41573,9,0.77179


## 7) Cosine similarity distribution plot

This histogram shows the distribution of cosine similarity scores across all precomputed
content-based recommendations. A right-skewed distribution (most scores near 0)
would indicate the embedding space is sparse. A concentrated distribution indicates
the autoencoder learned dense, meaningful geometry.

In [22]:
sim_values = content_recs_pl.filter(pl.col('rank') == 1)['cosine_similarity'].to_list()

fig_sim = go.Figure()
fig_sim.add_trace(go.Histogram(
    x=sim_values,
    nbinsx=60,
    marker_color='#6366f1',
    marker_line_color='#4338ca',
    marker_line_width=0.5,
    opacity=0.85,
    name='Top-1 cosine similarity',
))
fig_sim.update_layout(
    title='Distribution of top-1 cosine similarity (content-based recommender)',
    xaxis_title='Cosine similarity to nearest neighbor',
    yaxis_title='Number of query movies',
    height=450,
    template='plotly_white',
    bargap=0.05,
)
fig_sim.write_html(ARTIFACTS_DIR / 'week10_cosine_similarity_dist.html')
fig_sim.write_image(ARTIFACTS_DIR / 'week10_cosine_similarity_dist.png', scale=2)
fig_sim.show()

## 8) Popularity distribution plot

Log-scale histogram of rating counts across all catalog movies.
This is the long-tail view of the catalog that motivates the popularity baseline:
a small fraction of movies receive the overwhelming majority of ratings.

In [23]:
counts = movie_frame['rating_count'].to_list()

fig_pop = go.Figure()
fig_pop.add_trace(go.Histogram(
    x=np.log10(np.array(counts) + 1),
    nbinsx=50,
    marker_color='#10b981',
    marker_line_color='#059669',
    marker_line_width=0.5,
    opacity=0.85,
    name='log10(rating_count)',
))
fig_pop.update_layout(
    title='Rating count distribution across catalog (log10 scale)',
    xaxis_title='log10(rating count + 1)',
    yaxis_title='Number of movies',
    height=450,
    template='plotly_white',
    bargap=0.05,
)
fig_pop.write_html(ARTIFACTS_DIR / 'week10_popularity_distribution.html')
fig_pop.write_image(ARTIFACTS_DIR / 'week10_popularity_distribution.png', scale=2)
fig_pop.show()

## 9) Cluster-level popularity comparison

Bar chart showing the mean Bayesian score and mean rating count per cluster.
This visualizes how the cluster structure from Week 7 aligns with popularity signals.

In [24]:
cluster_stats = (
    popularity_global
    .group_by('cluster')
    .agg([
        pl.len().alias('n_movies'),
        pl.col('rating_count').mean().alias('mean_rating_count'),
        pl.col('avg_rating').mean().alias('mean_avg_rating'),
        pl.col('bayesian_score').mean().alias('mean_bayesian_score'),
    ])
    .sort('cluster')
)
cluster_stats.write_csv(ARTIFACTS_DIR / 'week10_cluster_popularity_stats.csv')

cluster_labels = [str(c) for c in cluster_stats['cluster'].to_list()]

fig_cluster = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Mean rating count per cluster', 'Mean Bayesian score per cluster'),
)
colors = px.colors.qualitative.Plotly

fig_cluster.add_trace(
    go.Bar(
        x=cluster_labels,
        y=cluster_stats['mean_rating_count'].to_list(),
        marker_color=[colors[i % len(colors)] for i in range(len(cluster_labels))],
        name='Mean rating count',
        showlegend=False,
    ),
    row=1, col=1,
)
fig_cluster.add_trace(
    go.Bar(
        x=cluster_labels,
        y=cluster_stats['mean_bayesian_score'].to_list(),
        marker_color=[colors[i % len(colors)] for i in range(len(cluster_labels))],
        name='Mean Bayesian score',
        showlegend=False,
    ),
    row=1, col=2,
)
fig_cluster.update_xaxes(title_text='Cluster', row=1, col=1)
fig_cluster.update_xaxes(title_text='Cluster', row=1, col=2)
fig_cluster.update_yaxes(title_text='Mean rating count', row=1, col=1)
fig_cluster.update_yaxes(title_text='Mean Bayesian score', row=1, col=2)
fig_cluster.update_layout(
    title='Cluster-level popularity statistics',
    height=450,
    template='plotly_white',
)
fig_cluster.write_html(ARTIFACTS_DIR / 'week10_cluster_popularity_stats.html')
fig_cluster.write_image(ARTIFACTS_DIR / 'week10_cluster_popularity_stats.png', scale=2)
fig_cluster.show()

## 10) Save baseline metadata

Save a JSON summary of the baseline configurations for reference in the evaluation notebook.

In [25]:
baseline_meta = {
    'catalog_size': movie_frame.height,
    'rated_movies': movie_frame.height,
    'global_mean_rating': round(global_mean, 6),
    'prior_count_bayesian': round(prior_count, 2),
    'embedding_dim': len(embedding_cols),
    'embedding_model': 'autoencoder_latent_13',
    'n_clusters': int(movie_frame['cluster'].n_unique()),
    'top_n_recs': TOP_N,
    'n_query_movies_content': N_QUERY_MOVIES,
    'baselines': [
        {'name': 'popularity_global',  'type': 'popularity', 'signal': 'bayesian_score'},
        {'name': 'popularity_cluster', 'type': 'popularity', 'signal': 'bayesian_score_within_cluster'},
        {'name': 'content_cosine',     'type': 'content',    'signal': 'cosine_similarity_ae13'},
    ],
    'artifacts': [
        'week10_popularity_global.csv',
        'week10_popularity_cluster.csv',
        'week10_content_recs_top20.parquet',
        'week10_cluster_popularity_stats.csv',
    ],
}

with open(ARTIFACTS_DIR / 'week10_baseline_meta.json', 'w') as f:
    json.dump(baseline_meta, f, indent=2)

print('Baseline metadata saved.')
print(json.dumps(baseline_meta, indent=2))

Baseline metadata saved.
{
  "catalog_size": 59047,
  "rated_movies": 59047,
  "global_mean_rating": 3.071374,
  "prior_count_bayesian": 1.0,
  "embedding_dim": 13,
  "embedding_model": "autoencoder_latent_13",
  "n_clusters": 7,
  "top_n_recs": 20,
  "n_query_movies_content": 5000,
  "baselines": [
    {
      "name": "popularity_global",
      "type": "popularity",
      "signal": "bayesian_score"
    },
    {
      "name": "popularity_cluster",
      "type": "popularity",
      "signal": "bayesian_score_within_cluster"
    },
    {
      "name": "content_cosine",
      "type": "content",
      "signal": "cosine_similarity_ae13"
    }
  ],
  "artifacts": [
    "week10_popularity_global.csv",
    "week10_popularity_cluster.csv",
    "week10_content_recs_top20.parquet",
    "week10_cluster_popularity_stats.csv"
  ]
}
